# Tutorial 11: Poisson LASSO

## Learning goals

After working through this notebook, you should be able to:

1. Prepare a model matrix for `glmnet`.
2. Use `cv.glmnet()` to tune a **Poisson LASSO** model with **cross-validated deviance**.
3. Extract `lambda.1se` and obtain the coefficients of the corresponding sparse model.
4. Identify which inputs are selected by the LASSO fit.
5. Use the selected model to predict counts in an independent test set.
6. Evaluate predictive performance on the test set.

In [ ]:
# Run this cell before continuing.
library(tidyverse)
library(glmnet)
library(ISLR2)

source("tests_tutorial_11.R")

## 1. The dataset

In this tutorial, we'll use the `Bikeshare`
[dataset](https://archive.ics.uci.edu/ml/datasets/bike+sharing+dataset) from the `ISLR2` package discussed in the [ISL book](https://www.statlearning.com)

The dataset contains information on the hourly usage of a bike-sharing program in Washington, DC, for different months, hours, weather conditions, and membership types (see `?Bikeshare`).

We will use LASSO penalization to build a strong predictive Poisson model and use it to predict rental counts in an independent test set.

Since the plain LASSO method is not particularly suited for categorical variables with many levels (and to avoid using group LASSO instead), we will first collapse several categorical variables into binary indicators:

- `is_weekend` vs weekday
- `bad_weather` vs good weather
- `peak_season` vs off-season
- `rush_hour` vs non-rush

We also keep the numeric predictors in the dataset.

The response variable is the hourly count of riders, `bikers`.

*Run the cells below.*

In [ ]:
data("Bikeshare", package = "ISLR2")

head(Bikeshare,2)

#### LASSO and categorical variables

When categorical variables have more than two levels, standard LASSO penalizes the coefficients of their dummy variables individually. As a result, LASSO may select some levels of a categorical variable but not others, rather than selecting or excluding the variable as a whole. 

To simplify variable selection and interpretation, we recode some categorical variables into meaningful binary variables.

In [ ]:
# Run this cell to create the processed data set.

bike_poisson_lasso <- 
  Bikeshare |>
  transmute(
    # retain the response and numeric
    bikers = bikers,
    temperature = temp,
    feel_temp = atemp,
    humidity = hum,	
    windspeed = windspeed,	  
    # create categorical variables with only 2 levels
    holiday = factor(holiday),  
    is_weekend = factor(
      if_else(weekday %in% c(0, 6), "weekend", "weekday")
    ),
    bad_weather = factor(
  if_else(
    weathersit %in% c("light rain/snow", "heavy rain/snow"),
    "bad","good")
    ),
    peak_season = factor(
      if_else(as.integer(mnth) %in% 5:10, "peak", "off")
    ),
    rush_hour = factor(
      if_else(hr %in% c("7", "8", "9", "17", "18"), "rush", "non_rush")
    )
  )

glimpse(bike_poisson_lasso)

In [ ]:
# Minimal EDA
summary(bike_poisson_lasso)
colSums(is.na(bike_poisson_lasso))

## 2. Split into training and test sets

We will use a 70/30 split, as in many of the other tutorial notebooks. 

Given that some categorical variables are not balanced, we'll make sure that the proportions are approximately maintained within groups defined by these variables, using `group_by()` and `sample_frac()`

We use a temporary row identifier to ensure that observations with identical values are not lost when constructing the test set.

In [ ]:
set.seed(2026)

bike_data <- bike_poisson_lasso |>
  mutate(id = row_number())

bike_train <- bike_data |>
  group_by(is_weekend, bad_weather, peak_season, rush_hour) |>
  sample_frac(0.70) |>
  ungroup()

bike_test <- bike_data |>
  anti_join(bike_train, by = "id")

bike_train <- bike_train |> select(-id)
bike_test <- bike_test |> select(-id)

dim(bike_train)
dim(bike_test)

## 3. Prepare the model matrix for `glmnet`

As before, `glmnet` needs the inputs in matrix form.
We remove the intercept column from the matrix returned by `model.matrix()`.


**Question 1.0**
<br>{points: 1}

Create a matrix with all predictors in `bike_train` using the `model.matrix()` function. 

> Recall that the first column of the resulting matrix is a column of 1s to model the intercept, which we remove because `glmnet` handles the unpenalized intercept separately.

Save the resulting matrix to an object called `model_matrix_X_train`.  

Also create the response matrix and save it as `matrix_Y_train`.

> Check Tutorial_09 if you don't remember how to do this.

*Fill out those parts indicated with `...`, uncomment the code, and run the cell.*

In [ ]:
# model_matrix_X_train <- 
#   ...(object = ..., data = ...)[, -1]

# matrix_Y_train <- 
#   ...(..., ncol = 1)

# your code here
fail() # No Answer - remove if you provide an answer

dim(model_matrix_X_train)
dim(matrix_Y_train)

In [ ]:
test_1.0()

## 4. Building a Poisson-LASSO model

We now want to build a predictive Poisson model using a **LASSO** penalty.

The steps are very similar to those used in multiple linear regression and logistic regression with a LASSO penalty. However, there are two important differences:

- You need to specify a `family` appropriate for count data.
- You need to choose an appropriate `type.measure` to select the level of penalization by cross-validation.

Here, we use **Poisson deviance** as the performance measure. It measures how well the predicted mean counts agree with the observed counts, accounting for the characteristics of Poisson distribution. 

Although **mean squared error (MSE)** is also an appropriate measure of predictive performance, Poisson deviance is more closely aligned with the distributional assumptions of the model. 

For both measures, smaller values indicate better predictive performance.

**Question 1.1**
<br>{points: 1}

We have already prepared our training data with `model_matrix_X_train` and `matrix_Y_train`. Now we need to select a value of $\lambda$ to build a **Poisson LASSO** model.

> The choice of $\lambda$ depends on the performance measure used in cross-validation. Check the available options for `type.measure` for the Poisson family.

Use `cv.glmnet()` with 5-fold cross-validation and Poisson deviance as the performance measure to select the level of penalization.

*Assign the output to `bike_cv_LASSO`.*

In [ ]:
set.seed(1234) # do not change this!

# bike_cv_LASSO <- 
#   ...(
#     x = ...,
#     y = ...,
#     alpha = ...,
#     family = "...",
#     type.measure = "...",
#     nfolds = ...
#   )

# bike_cv_LASSO

# your code here
fail() # No Answer - remove if you provide an answer

bike_cv_LASSO

In [ ]:
test_1.1()

### Selecting the level of penalization $\lambda$ 

The plot for `cv.glmnet()` shows the mean cross-validated deviance across the lambda grid, along with the vertical dotted lines for `lambda.min` and `lambda.1se`.


In [ ]:
options(repr.plot.width = 14, repr.plot.height = 8)

plot(
  bike_cv_LASSO,
  main = "Cross-Validation with Poisson LASSO\n\n"
)


**Question 1.2**
<br>{points: 1}

Using `bike_cv_LASSO`, obtain $\lambda_{1se}$ and assign it to the object `lambda_1se_deviance_LASSO`.

In [ ]:
# lambda_1se_deviance_LASSO <- ...

# your code here
fail() # No Answer - remove if you provide an answer

lambda_1se_deviance_LASSO


In [ ]:
test_1.2()

**Question 1.3**<br>{points: 1}

Get the estimated coefficients of the **Poisson-LASSO** model corresponding to a level of penalization `lambda = lambda_1se_deviance_LASSO`.

*Assign the estimated coefficients to an object called `coefs_bike_LASSO`.*

In [ ]:
# coefs_bike_LASSO <- ...

# your code here
fail() # No Answer - remove if you provide an answer

coefs_bike_LASSO

In [ ]:
test_1.3()

**Question 1.4**
<br>{points: 1}

Based on the coefficient output above, where coefficients equal to zero are shown as `.`, which input variables are selected by the Poisson-LASSO model built using the 1-SE rule with deviance as the performance metric?

**A.** `temperature`  
**B.** `feel_temp`  
**C.** `humidity`  
**D.** `windspeed`  
**E.** `holiday`  
**F.** `is_weekend`  
**G.** `bad_weather`  
**H.** `peak_season`  
**I.** `rush_hour`  

*Assign your answer to the object `answer1.4` as a single string containing the correct letters in **alphabetical order**. For example, if the selected variables were `temperature` and `holiday`, then you would write `"AE"`.*

In [ ]:
# answer1.4 <- ""

# your code here
fail() # No Answer - remove if you provide an answer

answer1.4

In [ ]:
test_1.4()

## 5. Evaluate the selected model on the test set

We can now use the Poisson-LASSO model built on the training data to predict the number of bike rentals in the test set.

**Question 1.5**
<br>{points: 1}

Let's start by creating the predictor matrix `model_matrix_X_test` and the response matrix `matrix_Y_test` from the test set.

*Fill out those parts indicated with ..., uncomment the corresponding code in the cell below, and run it.*

In [ ]:
# model_matrix_X_test <- ...

# matrix_Y_test <- ...

# your code here
fail() # No Answer - remove if you provide an answer

dim(model_matrix_X_test)
dim(matrix_Y_test)

In [ ]:
test_1.5()

**Question 1.6**
<br>{points: 1}

Using the Poisson-LASSO model selected and estimated with the 1-SE rule, generate predicted bike rental counts for the test set.

> Note that you can obtain predictions from the object `bike_cv_LASSO` by specifying the appropriate value of `s` and using `type = "response"` to obtain predicted mean counts.

Store the predicted values in an object named `bike_pred_test`.

*Fill out those parts indicated with ..., uncomment the corresponding code in the cell below, and run it.*

In [ ]:
# bike_pred_test <- ...(
#     ..., 
#     newx = ...,
#     type = "...",
#     s = "...")

# your code here
fail() # No Answer - remove if you provide an answer

head(bike_pred_test)

In [ ]:
test_1.6()

**Question 1.7**
<br>{points: 1}

We have obtained predicted counts for the test set, but how accurate are these predictions?

Calculate the **root mean squared error (RMSE)** between the observed and predicted bike rental counts in the test set. RMSE measures the typical magnitude of prediction errors in the original units of the response (bike rentals per hour), giving more weight to larger errors.

*Assign the result to `bike_rmse_test`, a single numeric value. Fill out the code below, uncomment it, and run the cell.*

In [ ]:
# bike_rmse_test <- ...

# your code here
fail() # No Answer - remove if you provide an answer

bike_rmse_test

In [ ]:
test_1.7()

### Discussion:

Interpret the RMSE in the context of bike rentals. 

- Would the model be useful for predicting hourly rental counts?
- What would you compare its RMSE with to assess whether its predictions are good?

The RMSE provides a measure of the magnitude of the prediction errors in the original units of the response variable (number of bike rentals per hour)

On its own, it is difficult to determine whether the RMSE is sufficiently small. As in previous worksheets and tutorials, we could compare the predictive performance of two or more models using their RMSE on the same test set. The model with the smallest test RMSE has better predictive performance according to this measure.